# Test Depth Anything V2 with your computer's webcam (Mac and Windows)

This notebook runs live depth estimation from your built-in webcam. You see your camera feed on the left and the predicted depth map on the right in a separate window.

**Before you start:** finish the setup notebook first. You need the `Depth-Anything-V2` repo cloned, its dependencies installed, and a checkpoint in `checkpoints/` (for example `depth_anything_v2_vits.pth`).

The video opens in its own window, so run this notebook locally (Jupyter or VS Code on your own computer). It will not work in a hosted notebook such as Google Colab, because those cannot reach your webcam.

## Step 1: Allow camera access

- **Mac:** the first run shows a camera permission prompt for the app running the notebook (Terminal, VS Code, etc.). Click **Allow**. If you missed it, go to **System Settings > Privacy & Security > Camera**, turn on that app, then restart it.
- **Windows:** go to **Settings > Privacy & security > Camera**. Turn on **Camera access** and **Let desktop apps access your camera**. Close other apps that use the webcam (Zoom, Teams, browsers).

## Step 2: Move into the repo folder

Run this only if the notebook is not already inside `Depth-Anything-V2`. Change the path to wherever you cloned the repo.

In [16]:
import os

# Adjust if needed, for example '../Depth-Anything-V2'
if not os.path.exists('depth_anything_v2'):
    %cd Depth-Anything-V2

print(os.getcwd())
print(os.listdir('checkpoints'))

c:\Users\afeta\eye-in-the-sky\Depth-Anything-V2
['depth_anything_v2_metric_hypersim_vits.pth', 'depth_anything_v2_vits.pth', 'student_unet.pth']


## Step 3: Create the webcam script

This cell writes `webcam_depth.py` into the repo folder. Edit `ENCODER` so it matches your checkpoint (`vits`, `vitb` or `vitl`).

In [20]:
%%writefile webcam_depth.py
import platform
import sys
import time

import cv2
import matplotlib
import numpy as np
import torch

MODEL = 'metric'      # 'teacher' = Depth Anything V2 (relative), 'metric' = its indoor metric version (distances in metres),
                       # 'student' = Level 2 U-Net (train it in Level2-Distill-Depth.ipynb).
ENCODER = 'vits'       # 'vits', 'vitb' or 'vitl'. Must match your checkpoint.
CAMERA_INDEX = 0       # 0 = default webcam. Try 1 if the wrong camera opens.
INPUT_SIZE = 308       # Lower = faster, higher = more detail. The default in the repo is 518.
METRIC_INPUT_SIZE = 518  # The metric model's distances shift a lot at other sizes, so keep it at its training size.
METRIC_SCALE = 1.0     # Metric only: multiply distances by this to calibrate (hold something at a known distance).
TOO_CLOSE_M = 0.5      # Metric only: anything in the flight path nearer than this many metres is an obstacle.
BLUR_KSIZE = 15        # Gaussian blur size (odd) before finding NEAR/FAR. Bigger = steadier, but thin objects (a finger) fade.
SMOOTH_ALPHA = 0.3     # Weight of the newest frame in the running average of the depth map. Lower = steadier but laggier.
FAR_PERCENT = 2        # FAR = centre of the largest patch among the farthest X% of pixels. Bigger = steadier, less precise.
GRID_ROWS, GRID_COLS = 6, 8  # Closeness grid, used by the overlay and by obstacle avoidance.
SHOW_GRID = True       # Draw the grid numbers on the camera feed.
GRID_STAT = 'mean'     # 'mean' = average closeness per cell (steady); 'max' = closest pixel per cell (won't hide a thin pole).
EDGE_MARGIN = 0.05     # Fraction of width/height ignored on each border when searching. 0 = search the whole frame.

# Obstacle avoidance. Depth is relative, so "too close" means: a big share of the flight path is much
# closer than the rest of the scene. Tune these live with your hand in front of the camera.
AVOID = True
PATH_W, PATH_H = 0.5, 0.6  # Flight-path box in the middle of the frame, as a fraction of width/height.
MIN_CONTRAST = 0.15    # If cells differ by less than this, the view is one flat surface and the grid can't be trusted.
CLOSE_LEVEL = 0.75     # Grid closeness (0-1) at which a cell counts as an obstacle.
BLOCK_FRACTION = 0.3   # TOO CLOSE when at least this share of the flight-path cells are obstacles.
TURN_PENALTY = 0.3     # Higher = prefer small course corrections over the most open cell.
ARROW_ALPHA = 0.3      # Smoothing of the arrow direction. Lower = steadier but slower to swing.
HOLD_SECONDS = 0.5     # Keep the warning up this long after the last detection, so it doesn't flicker.

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', DEVICE)

model_configs = {
    'vits': {'encoder': 'vits', 'features': 64,  'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

if MODEL == 'student':
    from student_unet import StudentDepth  # written by Level2-Distill-Depth.ipynb; same infer_image() call
    model = StudentDepth('checkpoints/student_unet.pth', DEVICE)
elif MODEL == 'metric':
    sys.path.insert(0, 'metric_depth')  # the metric model has its own copy of the depth_anything_v2 package
    from depth_anything_v2.dpt import DepthAnythingV2
    model = DepthAnythingV2(**model_configs[ENCODER], max_depth=20)  # 20 m = indoor (Hypersim) model
    model.load_state_dict(torch.load(f'checkpoints/depth_anything_v2_metric_hypersim_{ENCODER}.pth', map_location='cpu'))
    model = model.to(DEVICE).eval()
    INPUT_SIZE = METRIC_INPUT_SIZE
else:
    from depth_anything_v2.dpt import DepthAnythingV2
    model = DepthAnythingV2(**model_configs[ENCODER])
    model.load_state_dict(torch.load(f'checkpoints/depth_anything_v2_{ENCODER}.pth', map_location='cpu'))
    model = model.to(DEVICE).eval()

# On Windows the DirectShow backend opens the webcam faster and more reliably.
if platform.system() == 'Windows':
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
else:
    cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    sys.exit('Could not open the webcam. Check camera permissions and that no other app is using it.')

cmap = matplotlib.colormaps.get_cmap('Spectral_r')


def region_center(region):
    """Centre (x, y) of the largest connected patch in a boolean mask."""
    _, _, stats, centers = cv2.connectedComponentsWithStats(region.astype(np.uint8))
    biggest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])  # label 0 is the background
    return int(centers[biggest][0]), int(centers[biggest][1])


def depth_grid(depth, rows, cols, stat):
    """Per-cell closeness, rescaled so 0 = farthest cell and 1 = nearest cell in this frame.

    Also returns the contrast (spread between cells before rescaling): near 0 means the view is one
    flat surface, and rescaling has just stretched noise.
    """
    h, w = depth.shape
    reduce = np.max if stat == 'max' else np.mean
    grid = np.array([[reduce(depth[r * h // rows:(r + 1) * h // rows, c * w // cols:(c + 1) * w // cols])
                      for c in range(cols)] for r in range(rows)])
    contrast = grid.max() - grid.min()
    return (grid - grid.min()) / (contrast + 1e-8), contrast


def draw_grid(img, grid, metres=None):
    """Draw cell borders and values (metres if given, else 0-1 closeness), outlining the nearest cell red and the farthest cyan."""
    rows, cols = grid.shape
    h, w = img.shape[:2]
    cell = lambda r, c: ((c * w // cols, r * h // rows), ((c + 1) * w // cols, (r + 1) * h // rows))
    for r in range(rows):
        for c in range(cols):
            (x0, y0), (x1, y1) = cell(r, c)
            cv2.rectangle(img, (x0, y0), (x1, y1), (80, 80, 80), 1)
            text = f'{metres[r, c]:.1f}m' if metres is not None else f'{grid[r, c]:.1f}'
            cv2.putText(img, text, (x0 + 6, y0 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3)
            cv2.putText(img, text, (x0 + 6, y0 + 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    for idx, color in ((grid.argmax(), (0, 0, 255)), (grid.argmin(), (255, 255, 0))):
        cv2.rectangle(img, *cell(*np.unravel_index(idx, grid.shape)), color, 3)


def draw_marker(img, point, label, color):
    """Draw a crosshair with a filled label box at point (x, y)."""
    x, y = point
    cv2.circle(img, point, 12, (0, 0, 0), 4)
    cv2.circle(img, point, 12, color, 2)
    cv2.drawMarker(img, point, color, cv2.MARKER_CROSS, 30, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    # Keep the label inside the image near the edges.
    tx = min(max(x + 15, 0), img.shape[1] - tw - 6)
    ty = min(max(y - 15, th + 6), img.shape[0] - 6)
    cv2.rectangle(img, (tx - 3, ty - th - 5), (tx + tw + 3, ty + 5), color, -1)
    cv2.putText(img, label, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

def path_cells(rows, cols):
    """Boolean rows x cols mask of the cells inside the central flight-path box."""
    cy = (np.arange(rows) + 0.5) / rows - 0.5
    cx = (np.arange(cols) + 0.5) / cols - 0.5
    return (np.abs(cy)[:, None] <= PATH_H / 2) & (np.abs(cx)[None, :] <= PATH_W / 2)


def find_obstacles(grid, path, metres=None):
    """Obstacle cells, and whether the flight path is blocked.

    With metres (metric model): anything nearer than TOO_CLOSE_M, and any such cell in the path blocks it.
    Without (relative depth gives no distances): cells much closer than the rest of the scene, and the
    path is blocked when they cover BLOCK_FRACTION of it.
    """
    if metres is not None:
        obstacles = metres < TOO_CLOSE_M
        return obstacles, (obstacles & path).any()
    obstacles = grid >= CLOSE_LEVEL
    return obstacles, (obstacles & path).sum() >= BLOCK_FRACTION * path.sum()


def escape_direction(grid, obstacles):
    """Unit (dx, dy) from the frame centre toward the best open cell (dy < 0 = up), or None if all blocked."""
    rows, cols = grid.shape
    # A cell is only as open as its least open neighbour, so gaps right beside an obstacle score low.
    openness = cv2.erode((1 - grid).astype(np.float32), np.ones((3, 3), np.uint8))
    cy = ((np.arange(rows) + 0.5) / rows - 0.5)[:, None]
    cx = ((np.arange(cols) + 0.5) / cols - 0.5)[None, :]
    turn = np.sqrt(cx ** 2 + cy ** 2) / np.sqrt(0.5)  # 0 at the centre, 1 at the corners
    score = np.where(obstacles, -np.inf, openness - TURN_PENALTY * turn)
    if np.isinf(score).all():
        return None
    r, c = np.unravel_index(score.argmax(), score.shape)
    v = np.array([cx[0, c], cy[r, 0]])
    return v / np.linalg.norm(v)


def direction_label(d):
    """'GO UP-LEFT' style label for a direction (camera facing forward: up = climb, left = steer left)."""
    if d is None:
        return 'STOP'
    dx, dy = d
    vert = 'UP' if dy < -0.38 else 'DOWN' if dy > 0.38 else ''
    horiz = 'LEFT' if dx < -0.38 else 'RIGHT' if dx > 0.38 else ''
    return 'GO ' + '-'.join(part for part in (vert, horiz) if part)


def shade_cells(img, cells):
    """Tint the given grid cells red."""
    rows, cols = cells.shape
    h, w = img.shape[:2]
    shade = img.copy()
    for r, c in zip(*np.nonzero(cells)):
        cv2.rectangle(shade, (c * w // cols, r * h // rows), ((c + 1) * w // cols, (r + 1) * h // rows), (0, 0, 255), -1)
    cv2.addWeighted(shade, 0.3, img, 0.7, 0, img)


def draw_avoidance(img, path, status, direction, distance=None):
    """Draw the flight-path box, the status text and (when blocked) the escape arrow."""
    rows, cols = path.shape
    h, w = img.shape[:2]
    alert = status == 'TOO CLOSE'
    color = {'TOO CLOSE': (0, 0, 255), 'CLEAR': (0, 200, 0)}.get(status, (0, 140, 255))
    rs, cs = np.nonzero(path.any(1))[0], np.nonzero(path.any(0))[0]
    cv2.rectangle(img, (cs[0] * w // cols, rs[0] * h // rows), ((cs[-1] + 1) * w // cols, (rs[-1] + 1) * h // rows), color, 3)

    text = f'TOO CLOSE - {direction_label(direction)}' if alert else status
    if alert and distance is not None:
        text = f'TOO CLOSE {distance:.2f}m - {direction_label(direction)}'
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
    tx, ty = (w - tw) // 2, h - 20
    cv2.rectangle(img, (tx - 8, ty - th - 10), (tx + tw + 8, ty + 10), color, -1)
    cv2.putText(img, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    if alert and direction is not None:
        center = np.array([w // 2, h // 2])
        tip = tuple((center + direction * 0.3 * min(h, w)).astype(int))
        cv2.arrowedLine(img, tuple(center), tip, (0, 0, 0), 9, tipLength=0.3)
        cv2.arrowedLine(img, tuple(center), tip, (255, 255, 255), 4, tipLength=0.3)


path = path_cells(GRID_ROWS, GRID_COLS)
last_alert = -float('inf')
avoid_dir = None    # smoothed escape direction, None when clear or fully blocked

prev = time.time()
avg_depth = None    # running (time-averaged) depth map
avg_metres = None   # metric only: running distance map in metres
search_mask = None  # 255 where NEAR/FAR may be found, 0 on the ignored border

print('Press q in the video window to quit.')
while True:
    ok, frame = cap.read()
    if not ok:
        print('Failed to read a frame from the webcam.')
        break

    depth = model.infer_image(frame, INPUT_SIZE).astype(np.float32)

    # Metric: keep the distances, and hand the rest of the code closeness (1 / distance) so that,
    # like the relative models, higher = closer and the Level 1 code works unchanged.
    if MODEL == 'metric':
        metres = cv2.GaussianBlur(np.maximum(depth * METRIC_SCALE, 0.05), (BLUR_KSIZE, BLUR_KSIZE), 0)
        avg_metres = metres if avg_metres is None else SMOOTH_ALPHA * metres + (1 - SMOOTH_ALPHA) * avg_metres
        depth = 1.0 / np.maximum(depth * METRIC_SCALE, 0.05)

    # Level 1: nearest and farthest points.
    # Depth Anything V2 outputs relative inverse depth, so higher = closer:
    # the max is the NEAREST point and the min is the FARTHEST.
    # 1. Blur so single noisy pixels can't win.
    # 2. Scale to 0-1: raw values aren't comparable between frames.
    # 3. Average over time so near-ties (e.g. a flat back wall) don't flip every frame.
    # 4. Search inside the mask, skipping the unreliable border.
    # NEAR = single closest pixel (keeps small close objects like a finger).
    # FAR = centre of the biggest far region (a flat wall has thousands of near-tied
    #       pixels, so the single farthest one hops around; the region's centre doesn't).
    smooth = cv2.GaussianBlur(depth, (BLUR_KSIZE, BLUR_KSIZE), 0)
    smooth = (smooth - smooth.min()) / (smooth.max() - smooth.min() + 1e-8)
    if avg_depth is None:
        avg_depth = smooth
        h, w = smooth.shape
        mx, my = int(w * EDGE_MARGIN), int(h * EDGE_MARGIN)
        search_mask = np.zeros((h, w), np.uint8)
        search_mask[my:h - my, mx:w - mx] = 255
    else:
        avg_depth = SMOOTH_ALPHA * smooth + (1 - SMOOTH_ALPHA) * avg_depth
    _, _, _, near_pt = cv2.minMaxLoc(avg_depth, search_mask)
    inside = search_mask > 0
    far_cut = np.percentile(avg_depth[inside], FAR_PERCENT)
    far_pt = region_center((avg_depth <= far_cut) & inside)

    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8) * 255.0
    depth = depth.astype(np.uint8)
    depth_color = (cmap(depth)[:, :, :3] * 255)[:, :, ::-1].astype(np.uint8)

    now = time.time()
    fps = 1.0 / max(now - prev, 1e-6)
    prev = now
    cv2.putText(depth_color, f'{MODEL} {fps:.1f} FPS', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    grid, contrast = depth_grid(avg_depth, GRID_ROWS, GRID_COLS, GRID_STAT)
    cell_m = None  # metric only: distance to the closest point in each cell
    if MODEL == 'metric':
        h, w = avg_metres.shape
        cell_m = np.array([[avg_metres[r * h // GRID_ROWS:(r + 1) * h // GRID_ROWS, c * w // GRID_COLS:(c + 1) * w // GRID_COLS].min()
                            for c in range(GRID_COLS)] for r in range(GRID_ROWS)])

    # Obstacle avoidance: warn when the flight path is blocked and point toward open space.
    if AVOID:
        flat = cell_m is None and contrast < MIN_CONTRAST  # one surface fills the view: relative depth can't say how close it is
        obstacles, blocked = find_obstacles(grid, path, cell_m)
        if flat:
            obstacles, blocked = np.zeros_like(obstacles), False
        if blocked:
            last_alert = now
            new_dir = escape_direction(grid, obstacles)
            if new_dir is None or avoid_dir is None:
                avoid_dir = new_dir
            else:
                avoid_dir = ARROW_ALPHA * new_dir + (1 - ARROW_ALPHA) * avoid_dir
                avoid_dir = avoid_dir / (np.linalg.norm(avoid_dir) + 1e-8)
        alert = now - last_alert < HOLD_SECONDS
        if not alert:
            avoid_dir = None
        status = 'TOO CLOSE' if alert else 'NO DEPTH CONTRAST' if flat else 'CLEAR'
        shade_cells(frame, obstacles)
    if SHOW_GRID:
        draw_grid(frame, grid, cell_m)
    if AVOID:
        draw_avoidance(frame, path, status, avoid_dir, cell_m[path].min() if cell_m is not None else None)

    near_label, far_label = 'NEAR', 'FAR'
    if MODEL == 'metric':
        near_label = f'NEAR {avg_metres[near_pt[1], near_pt[0]]:.2f}m'
        far_label = f'FAR {avg_metres[far_pt[1], far_pt[0]]:.1f}m'
    for img in (frame, depth_color):
        draw_marker(img, near_pt, near_label, (0, 0, 255))    # red (BGR)
        draw_marker(img, far_pt, far_label, (255, 255, 0))    # cyan (BGR)

    cv2.imshow('Depth Anything V2 - webcam (left) and depth (right). Press q to quit.',
               np.hstack([frame, depth_color]))
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Overwriting webcam_depth.py


## Step 4: Run it

This opens a window with your webcam on the left and the depth map on the right. By default, warm colors (red) are close to the camera and cool colors (blue) are far away. A red **NEAR** marker shows the closest point in the scene and a cyan **FAR** marker the farthest, on both the camera feed and the depth map. The box in the middle of the camera feed is the drone's flight path. When something close blocks enough of it, the box turns red, close cells are shaded, and an arrow and label (for example **TOO CLOSE - GO LEFT**) point toward open space; otherwise it reads **CLEAR**. Tune it with the avoidance settings at the top of Step 3. Click the video window and press **q** to quit. The cell keeps running until you do.

The cell uses the same Python as this notebook, so it works on both Mac and Windows.

In [22]:
import sys

!{sys.executable} webcam_depth.py

Using device: cuda
Press q in the video window to quit.


xFormers not available
xFormers not available


## Step 5: Try it out

- Hold your hand close to the camera, then move it away. The colors should shift.
- Walk back from the camera and watch your body change color.
- Point the camera at a room to see walls, furniture and doorways at different depths.
- Depth is **relative**, not in meters. The colors show what is nearer or farther, not exact distance.

## Tuning speed

Edit the settings at the top of the script cell in Step 3, run that cell again, then re-run Step 4.

| Setting | Change | Effect |
|---|---|---|
| `ENCODER` | `vits` (fastest), `vitb`, `vitl` (slowest) | Larger models are more accurate but slower. The encoder must match the checkpoint you downloaded. |
| `INPUT_SIZE` | Lower (for example 252) or higher (518) | Lower is faster. 518 is the repo default. |
| Device | NVIDIA GPU (Windows) or Apple Silicon (Mac) | Much faster than CPU. The script picks the best one automatically. |

On a CPU-only machine, expect a low frame rate even with the Small model. That is normal.

## Troubleshooting

- **"Could not open the webcam":** check the permissions in Step 1 and close other apps using the camera. Try `CAMERA_INDEX = 1`.
- **Wrong camera opens (for example a phone via Continuity Camera on Mac):** change `CAMERA_INDEX` to 0 or 1.
- **Black or frozen window on Windows:** make sure the `cv2.CAP_DSHOW` line is in the script, or try removing it.
- **Very low FPS:** lower `INPUT_SIZE`, use `vits`, and close other heavy apps. On Windows with an NVIDIA GPU, confirm you installed the CUDA build of PyTorch. The script prints the device it uses at startup, and `cpu` there means no GPU is being used.
- **`FileNotFoundError` for the .pth file:** the checkpoint is not in `checkpoints/`, or `ENCODER` does not match the file name.
- **`ModuleNotFoundError: depth_anything_v2`:** the notebook is not inside the `Depth-Anything-V2` folder. Re-run Step 2.
- **OpenCV window does not appear on Mac:** if you installed `opencv-python-headless`, uninstall it and run `%pip install opencv-python`.
- **Notebook seems stuck after pressing q:** the cell stops when the video window closes. If it does not, use the notebook's Interrupt button.

## Links

- Code: https://github.com/DepthAnything/Depth-Anything-V2
- Models: https://huggingface.co/depth-anything